# TAB 68B: exp68b_spectral — EXP-68B (unified latent + spectral c_B + AdaLN + normalized L1) - Colab
# Delta so với EXP-68A (và so với bản exp68 đã commit):
#   1. --unified_latent true, --latent_disentanglement false: 1 latent 64D thay 2 subspace.
#   2. --spectral_condition_token true + --spectral_phase_encoding cos_sin: token 24-D gồm
#      (w_k cos theta_k, w_k sin theta_k) và E_k/sumE, w_k = sqrt(E_k/sumE).
#      - Không còn gián đoạn tại +-pi (nhánh raw theta/pi nhảy 2.0 khi lật dấu).
#      - Bin có năng lượng ~0 (pha là nhiễu số học) bị nhân bởi w_k ~ 0 => không bơm nhiễu.
#      - Đo trên 81 file: token tách waveform (L2 2.82) và sensor (2.22) nhưng gần như mù
#        specimen (0.28) và liftoff (0.18) => đúng nghĩa "condition token".
#   3. --condition_modulation adaln + cond_in được L2-normalize trước AdaLN (trainer cũ truyền
#      thô => gamma/beta không chặn => c_B thành "bias bank" từng scan: đây là cơ chế thật của
#      triệu chứng "dùng c_B là học cực nhanh").
#   4. --cov_weight 0.2 (full-rank 64D VICReg, anti_hijack_ratio 0).
#   5. --ref_patch_mode corner_median: patch nền [10:15,10:15] của bản commit chứa 96% pixel
#      khuyết tật trên Corrosion (đo trên GT) => c_B đang là dấu vân tay khuyết tật chứ không phải
#      hàm truyền. corner_median = trung vị 4 góc: 0% pixel khuyết tật, cách khuyết tật >= 8.5 px.
# Bar: test LP phải lệch > 0.04 so exp63 (0.6435) mới được coi là có tác dụng thật.

# VẬN HÀNH (đã sửa): `runs/` được symlink sang Drive + assert kiểm chứng; cell cuối copy
# dự phòng nếu symlink hỏng. Preflight kiểm tra `train.py` có đủ flag trước khi train
# (lỗi "unrecognized arguments" của 68B là do clone Colab thiếu code mới).

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os, glob
DATA = '/content/drive/MyDrive/PECT_Dataset/'
DRIVE_RUNS = '/content/drive/MyDrive/PECT_Dataset/runs/'
os.makedirs(DRIVE_RUNS, exist_ok=True)
# Verify: TDMS + GT masks phai thay tu Drive
tdms = [f for f in glob.glob(os.path.join(DATA, '**', '*.tdms'), recursive=True) if not f.endswith('_index.tdms')]
print('TDMS:', len(tdms))
for p in [os.path.join(DATA, 'ground_truth/corrosion/corrosion_gt_mask.npy'),
          os.path.join(DATA, 'ground_truth/rivet_v1/rivet_v1_gt_mask.npy'),
          os.path.join(DATA, 'ground_truth/rivet_v2/rivet_v2_gt_mask.npy')]:
    print(('OK  ' if os.path.exists(p) else 'MISS') + ' ' + p)

In [ ]:
!test -d GraduationProject && (cd GraduationProject && git fetch origin && git checkout Define_PECT_JEPA && git reset --hard origin/Define_PECT_JEPA) || git clone -b Define_PECT_JEPA https://github.com/TranAnh35/GraduationProject.git
!cd GraduationProject && git log --oneline -3

In [ ]:
# [FIX] sklearn version co sample_weight trong MLPClassifier.fit -> ep dung 1.7.2 (giong env lab)
!pip install -q -U "scikit-learn==1.7.2" nptdms scipy tqdm 2>&1 | tail -2
!cd GraduationProject && python -c "import torch, sklearn, scipy, nptdms; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(), '| sklearn:', sklearn.__version__)"

In [ ]:
# ==============================================================================
# PREFLIGHT: code tren Colab PHAI la ban moi nhat. Neu thieu flag, train.py se chet voi
# "error: unrecognized arguments" (da xay ra voi EXP-68B vi clone cu chua co flag moi).
# ==============================================================================
import subprocess

REQUIRED_FLAGS = ['--spectral_condition_token', '--spectral_phase_encoding',
                  '--ref_patch_mode', '--unified_latent', '--pred_loss_type',
                  '--self_focus_gamma', '--num_target_rays']

TRAIN_PY = 'GraduationProject/src/PECT_JEPA/cross_condition/train.py'
src = open(TRAIN_PY, encoding='utf-8').read()
missing = [f for f in REQUIRED_FLAGS if f not in src]

head = subprocess.run(['git', '-C', 'GraduationProject', 'rev-parse', '--short', 'HEAD'],
                      capture_output=True, text=True).stdout.strip()
branch = subprocess.run(['git', '-C', 'GraduationProject', 'rev-parse', '--abbrev-ref', 'HEAD'],
                        capture_output=True, text=True).stdout.strip()
print(f'[preflight] branch={branch} HEAD={head}')
print(f'[preflight] flags: {"OK" if not missing else "THIEU " + str(missing)}')
if missing:
    raise SystemExit(
        'train.py tren Colab thieu flag: ' + ', '.join(missing) +
        '\n-> Chay lai cell git o tren (git pull origin Define_PECT_JEPA) roi chay lai cell nay. '
        'KHONG chay cell Train khi con thieu flag.')

In [ ]:
# ==============================================================================
# Gan experiments/cross_condition/runs -> Google Drive
# Checkpoint, metrics_epoch.csv, run.log ghi truc tiep len Drive (khong mat khi Colab ngat).
# Cell tu kiem chung: assert se dung notebook neu symlink khong tro ve Drive.
# ==============================================================================
import os, shutil, subprocess

RUNS_LINK = 'GraduationProject/experiments/cross_condition/runs'
os.makedirs(DRIVE_RUNS, exist_ok=True)
os.makedirs(os.path.dirname(RUNS_LINK), exist_ok=True)

if os.path.islink(RUNS_LINK):
    print('ALREADY_LINKED ->', os.path.realpath(RUNS_LINK))
else:
    if os.path.isdir(RUNS_LINK):
        for name in os.listdir(RUNS_LINK):
            s, d = os.path.join(RUNS_LINK, name), os.path.join(DRIVE_RUNS, name)
            if not os.path.exists(d):
                shutil.copytree(s, d)
                print('  migrated', name)
        shutil.rmtree(RUNS_LINK)
    os.symlink(DRIVE_RUNS, RUNS_LINK)
    print('LINKED', RUNS_LINK, '->', DRIVE_RUNS)

assert os.path.realpath(RUNS_LINK) == os.path.realpath(DRIVE_RUNS), \
    f'Symlink KHONG tro ve Drive: {os.path.realpath(RUNS_LINK)}'
print(subprocess.run(['ls', '-la', os.path.dirname(RUNS_LINK)],
                     capture_output=True, text=True).stdout)

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

# ===== EXP-68B CONFIG: Unified 64D + spectral c_B (cos/sin) + AdaLN + normalized L1 =====
EXP_NAME = 'exp68b_spectral'

BASE_FLAGS = '--star_radii 2 4 7 --tokenizer_type energy_adaptive --latent_disentanglement false --unified_latent true --two_stream_backbone false --embed_dim 64 --var_weight 0.5 --cov_weight 0.2 --var_gamma 1.0 --scan_centered_vicreg true --inv_start_epoch 11 --inv_ramp_epochs 20 --inv_cosine true --scan_centered_inv true --inv_weight 0.2 --diverge_patience 0 --device cuda'
EXTRA = '--continuous_pos_embed true --contiguous_masking true --relative_energy_anchor true --bi_directional_relative_energy true --patch_polarity_align true --spectral_condition_token true --spectral_phase_encoding cos_sin --use_condition_token true --condition_modulation adaln --pred_loss_type normalized_l1 --anti_hijack_ratio 0.0 --self_focus_gamma 0.0 --ref_patch_mode corner_median'

os.environ.update({'EXP_NAME': EXP_NAME, 'BASE_FLAGS': BASE_FLAGS, 'EXTRA': EXTRA, 'DATA': DATA, 'DRIVE_RUNS': DRIVE_RUNS})
print('EXPERIMENT:', EXP_NAME)
print('FLAGS:', BASE_FLAGS + ' ' + EXTRA)

In [ ]:
# Cell Train (Fresh Run): exp68b_spectral — unified latent + spectral c_B + AdaLN + normalized L1
!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.train --exp_name $EXP_NAME --add_timestamp true $BASE_FLAGS $EXTRA --data_dir "$DATA" --epochs 100 --eval_val_every 1

In [ ]:
# ==============================================================================
# Xac nhan run folder DA NAM TREN DRIVE. Neu run dir khong nam trong DRIVE_RUNS
# (symlink hong / bi thay bang folder that) thi copy sang Drive ngay tai day.
# ==============================================================================
import glob, os, shutil

cands = sorted(glob.glob(os.path.join(DRIVE_RUNS, f'{EXP_NAME}_*')), key=os.path.getmtime)
run_dir = os.environ.get('LAST_RUN_DIR') or (cands[-1] if cands else None)
if not run_dir or not os.path.isdir(run_dir):
    raise FileNotFoundError(f'Khong thay run {EXP_NAME} tren Drive: {DRIVE_RUNS}')

real_run = os.path.realpath(run_dir)
real_drive = os.path.realpath(DRIVE_RUNS)
if not real_run.startswith(real_drive):
    dst = os.path.join(DRIVE_RUNS, os.path.basename(real_run))
    shutil.copytree(real_run, dst, dirs_exist_ok=True)
    print('[copy] run dir KHONG nam tren Drive -> da copy sang', dst)
    run_dir = dst

print('[*] run dir tren Drive:', run_dir)
for f in ['config.json', 'run.log', 'metrics_epoch.csv',
          'checkpoints/best_cross_condition_jepa.pt',
          'checkpoints/latest_cross_condition_jepa.pt']:
    p = os.path.join(run_dir, f)
    print(('  OK   ' if os.path.exists(p) else '  MISS ') + f)
print('[*] entries:', sorted(os.listdir(run_dir)))

In [ ]:
# ==============================================================================
# Resume training (CHI chay khi session truoc da dung han).
# Bat buoc truyen lai $BASE_FLAGS $EXTRA: train.py dung checkpoint de suy ra run_dir,
# nhung config/architecture van duoc dung tu flag -> thieu flag = sai kien truc, load_state_dict fail.
# ==============================================================================
import glob, os

SPECIFIC_RUN = ''  # de trong = run moi nhat cua EXP_NAME

cands = sorted(glob.glob(os.path.join(DRIVE_RUNS, f'{EXP_NAME}_*')), key=os.path.getmtime)
if not cands:
    raise FileNotFoundError(f'Khong thay run {EXP_NAME} trong {DRIVE_RUNS}')
run_dir = os.path.join(DRIVE_RUNS, SPECIFIC_RUN) if SPECIFIC_RUN else cands[-1]

ck_latest = os.path.join(run_dir, 'checkpoints', 'latest_cross_condition_jepa.pt')
ck_best = os.path.join(run_dir, 'checkpoints', 'best_cross_condition_jepa.pt')
if not (os.path.exists(ck_latest) or os.path.exists(ck_best)):
    raise FileNotFoundError(f'Khong co checkpoint trong {os.path.join(run_dir, "checkpoints")}')
resume_ckpt = ck_latest if os.path.exists(ck_latest) else ck_best
print('[*] run_dir    :', run_dir)
print('[*] resume ckpt:', resume_ckpt, f'({os.path.getsize(resume_ckpt)/1024**2:.1f} MB)')
os.environ['RESUME_CKPT'] = resume_ckpt
os.environ['LAST_RUN_DIR'] = run_dir

!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.train --exp_name $EXP_NAME $BASE_FLAGS $EXTRA --data_dir "$DATA" --epochs 100 --eval_val_every 1 --resume_checkpoint "$RESUME_CKPT"

In [ ]:
# ==============================================================================
# Xac dinh run folder (moi nhat cua EXP_NAME) + checkpoint de eval/downstream
# ==============================================================================
import glob, os

SPECIFIC_RUN = ''  # de trong = run moi nhat

cands = sorted(glob.glob(os.path.join(DRIVE_RUNS, f'{EXP_NAME}_*')), key=os.path.getmtime)
if not cands:
    raise FileNotFoundError(f'Khong thay run {EXP_NAME} trong {DRIVE_RUNS}. Chay cell Train truoc.')
run_dir = os.path.join(DRIVE_RUNS, SPECIFIC_RUN) if SPECIFIC_RUN else cands[-1]
os.environ['LAST_RUN_DIR'] = run_dir

ck_best = os.path.join(run_dir, 'checkpoints', 'best_cross_condition_jepa.pt')
ck_latest = os.path.join(run_dir, 'checkpoints', 'latest_cross_condition_jepa.pt')
if not (os.path.exists(ck_best) or os.path.exists(ck_latest)):
    raise FileNotFoundError(f'Khong co checkpoint trong {os.path.join(run_dir, "checkpoints")}')
eval_ckpt = ck_best if os.path.exists(ck_best) else ck_latest
os.environ['EVAL_CKPT'] = eval_ckpt
print('[*] run_dir :', run_dir)
print('[*] ckpt    :', eval_ckpt, f'({os.path.getsize(eval_ckpt)/1024**2:.1f} MB)')

In [ ]:
# ==============================================================================
# (Tuy chon) Train downstream: Linear/MLP probes + Sizing + 7 model UDA
# --adaptation_mode all = 4 canonical single-variable shifts (8 file/shift).
# KHONG dung "all_test" (adapt tren ca 65 file test = transductive, xem _audit/REPORT_UDA_DanhGia.md).
# ==============================================================================
!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.downstream.train_downstream --checkpoint "$EVAL_CKPT" --data_dir "$DATA" --adaptation_mode all --device cuda --epochs 30 --uda_batch_size 128 --learning_rate 1e-3

In [ ]:
# ==============================================================================
# Full downstream evaluation (65 test scans: Linear Probe, Mahalanobis, Sizing, UDA neu co)
# + readout diagnostics
# ==============================================================================
!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.evaluate --checkpoint "$EVAL_CKPT" --data_dir "$DATA" --device cuda
!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.diagnose_readouts --checkpoint "$EVAL_CKPT" --data_dir "$DATA" --device cuda